In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import os

ICEBERG_VERSION = "1.5.0"
DEPENDENCIES = f"org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:{ICEBERG_VERSION}"

# Configurando a SparkSession com suporte duplo (Delta + Iceberg)
builder = SparkSession.builder \
    .appName("Trabalho-Eng-Dados-Isaac") \
    .config("spark.jars.packages", DEPENDENCIES) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension,org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkSessionCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hive") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "warehouse")

# Inicializa a sessão aplicando as configurações do Delta
spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("✅ Sucesso! Spark Session ativa com suporte a Delta Lake e Apache Iceberg.")

✅ Sucesso! Spark Session ativa com suporte a Delta Lake e Apache Iceberg.


In [12]:
import shutil
import os

# 1. Definindo os dados iniciais
dados_vendas = [
    (1, "Laptop Gamer", 5500.0, 5),
    (2, "Mouse Sem Fio", 120.0, 20),
    (3, "Teclado Mecanico", 250.0, 15),
    (4, "Monitor 4K", 1800.0, 8),
    (5, "Cabo HDMI 2.1", 75.0, 50)
]

schema = ["id", "produto", "preco", "estoque"]
df_inicial = spark.createDataFrame(dados_vendas, schema)

# 2. Limpeza para garantir que o Notebook rode sem erros de conflito
caminho_delta = "warehouse/tabela_vendas_delta"
if os.path.exists(caminho_delta):
    shutil.rmtree(caminho_delta)

# 3. Salvando em Delta Lake
print("💾 Salvando dados iniciais no Delta Lake...")
df_inicial.write.format("delta").mode("overwrite").save(caminho_delta)

print("\n✅ ESTADO INICIAL:")
df_inicial.show()

💾 Salvando dados iniciais no Delta Lake...



✅ ESTADO INICIAL:


+---+----------------+------+-------+
| id|         produto| preco|estoque|
+---+----------------+------+-------+
|  1|    Laptop Gamer|5500.0|      5|
|  2|   Mouse Sem Fio| 120.0|     20|
|  3|Teclado Mecanico| 250.0|     15|
|  4|      Monitor 4K|1800.0|      8|
|  5|   Cabo HDMI 2.1|  75.0|     50|
+---+----------------+------+-------+



In [13]:
from delta.tables import DeltaTable

print("🛠️ Aplicando aumento de preço no Delta Lake...")

# Carrega a tabela Delta para manipulação
delta_table = DeltaTable.forPath(spark, "warehouse/tabela_vendas_delta")

# Atualiza o preço do ID 1
delta_table.update(
    condition="id = 1", 
    set={"preco": "6000.0"}
)

print("\n✅ ESTADO APÓS UPDATE:")
# Lê a tabela atualizada para mostrar a mudança
spark.read.format("delta").load("warehouse/tabela_vendas_delta").filter("id = 1").show()

🛠️ Aplicando aumento de preço no Delta Lake...



✅ ESTADO APÓS UPDATE:
+---+------------+------+-------+
| id|     produto| preco|estoque|
+---+------------+------+-------+
|  1|Laptop Gamer|6000.0|      5|
+---+------------+------+-------+



In [14]:
print("🗑️ Removendo o produto 'Cabo HDMI 2.1' (ID 5)...")

# Localiza e deleta o registro
delta_table = DeltaTable.forPath(spark, "warehouse/tabela_vendas_delta")
delta_table.delete(condition="id = 5")

print("\n✅ ESTADO FINAL APÓS DELETE:")
# Mostra a tabela final sem o item 5
spark.read.format("delta").load("warehouse/tabela_vendas_delta").show()

🗑️ Removendo o produto 'Cabo HDMI 2.1' (ID 5)...



✅ ESTADO FINAL APÓS DELETE:
+---+----------------+------+-------+
| id|         produto| preco|estoque|
+---+----------------+------+-------+
|  3|Teclado Mecanico| 250.0|     15|
|  2|   Mouse Sem Fio| 120.0|     20|
|  1|    Laptop Gamer|6000.0|      5|
|  4|      Monitor 4K|1800.0|      8|
+---+----------------+------+-------+



# Trabalho de Engenharia de Dados: Delta Lake & Apache Iceberg
**Professor:** Jorge Luiz Silva
**Aluno:** Isaac Alexsander

### Cenário do Projeto
O objetivo deste notebook é demonstrar a implementação de tabelas transacionais utilizando Apache Spark. 
Utilizamos uma fonte de dados de **Vendas** para realizar operações de INSERT, UPDATE e DELETE, evidenciando as características de tabelas Delta e Iceberg.

### Modelo Entidade-Relacionamento (ER) Simples
- **Tabela:** `vendas`
- **Campos:** `id` (PK), `produto`, `preco`, `estoque`.